# The Chat Format

In this notebook, you will explore how you can utilize the chat format to have extended conversations with chatbots personalized or specialized for specific tasks or behaviors.

## Setup

In [1]:
import os
import openai
from dotenv import load_dotenv, find_dotenv
# _ = load_dotenv(find_dotenv()) # read local .env file

env_path="d:/code/keys/.env"
_=load_dotenv(env_path)
# read local .env file(环境变量文件)

openai.api_key  = os.getenv('OPENAI_API_KEY')

In [2]:
# old helper function
def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message["content"]

# 定义新的helper function, there are more roles in message
def get_completion_from_messages(messages, model="gpt-3.5-turbo", temperature=0):
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=temperature, # this is the degree of randomness of the model's output
    )
#     print(str(response.choices[0].message))
    return response.choices[0].message["content"]

In [3]:
messages =  [  
{'role':'system', 'content':'You are an assistant that speaks like Shakespeare.'},    
{'role':'user', 'content':'tell me a joke'},   
{'role':'assistant', 'content':'Why did the chicken cross the road'},   
{'role':'user', 'content':'I don\'t know'}  ]

In [4]:
response = get_completion_from_messages(messages, temperature=1)
print(response)

Why, to traverse the perilous path to the other side, of course!


In [5]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Hi, my name is Isa'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Hello Isa! How can I assist you today?


In [6]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},    
{'role':'user', 'content':'Yes,  can you remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

I apologize, but I don't have access to personal information about individuals unless it has been shared with me in the course of our conversation. I value and respect your privacy. My primary function is to provide information and assist with tasks to the best of my abilities.


In [7]:
messages =  [  
{'role':'system', 'content':'You are friendly chatbot.'},
{'role':'user', 'content':'Hi, my name is Isa'},
{'role':'assistant', 'content': "Hi Isa! It's nice to meet you. \
Is there anything I can help you with today?"},
{'role':'user', 'content':'Yes, you can remind me, What is my name?'}  ]
response = get_completion_from_messages(messages, temperature=1)
print(response)

Your name is Isa.


# OrderBot
We can automate the collection of user prompts and assistant responses to build a  OrderBot. The OrderBot will take orders at a pizza restaurant. 

In [8]:
def collect_messages(_):
    prompt = inp.value_input
    # 从一个名为inp的输入控件中获取用户输入的消息，并将其存储在prompt变量中。
    inp.value = ''
    # 将inp的值清空，以准备接受下一个用户输入。
    context.append({'role':'user', 'content':f"{prompt}"})
    # 将用户的消息（prompt）以字典形式添加到名为context的列表中。该字典包含两部分信息：消息的角色（"user"表示用户）和消息内容（即用户的输入）。
    response = get_completion_from_messages(context) 
    # 获取助手的响应
    context.append({'role':'assistant', 'content':f"{response}"})
    # 将助手的响应（response）以字典形式添加到context列表中
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    # 这是Panel库中的一个函数，于创建一个包含多个元素的水平行,。其中包含两个元素。1)User 2) 创建一个Markdown格式的文本面板，其中包含用户的消息。prompt是之前从用户输入中获取的消息。width=600表示设置面板的宽度为600像素。
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))
    # 创建一个水平行，其中包含两个元素。1)Assistant 2) 创建一个Markdown格式的文本面板，其中包含助手的响应，response是从模型获得的。width=600表示设置面板的宽度为600像素，style={'background-color': '#F6F6F6'}用于设置面板的背景颜色。
 
    return pn.Column(*panels)


In [13]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages

# 以下代码创建了一个基于Panel库的交互式对话界面，它允许用户输入文本消息并与一个名为button_conversation的按钮进行交互，以触发对话的进行和展示。
inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
# 创建了一个文本输入框控件（TextInput），称为inp。它的默认值设置为"Hi"，并且具有一个占位符文本，当用户没有输入时，显示 "Enter text here…"。这个控件用于用户输入消息。
button_conversation = pn.widgets.Button(name="Chat!")
# 创建了一个按钮控件（Button），称为button_conversation。按钮的标签（文本）设置为 "Chat!"。这个按钮用于触发对话的进行。
interactive_conversation = pn.bind(collect_messages, button_conversation)
# 使用pn.bind函数创建了一个可交互的对话。它将collect_messages函数与button_conversation按钮绑定在一起，以便在用户点击按钮时触发collect_messages函数并更新对话。

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)
# 创建了一个名为dashboard的列，将各个控件和对话部分组合到一起以构建完整的用户界面。
# inp 控件被添加到列中，用于用户输入消息。
# pn.Row(button_conversation) 创建一个包含按钮的水平行，并将它添加到列中，以便用户可以点击按钮来触发对话。
# pn.panel(interactive_conversation, loading_indicator=True, height=300) 创建一个包含交互式对话的面板，并将其添加到列中。这个面板使用了interactive_conversation来显示对话，并启用了加载指示器（loading indicator），这是在加载对话时显示的可视标识，以及设置了面板的高度为300像素。
dashboard

C:\Users\wumin\AppData\Local\Temp\ipykernel_21240\1943865245.py:16: PanelDeprecationWarning: 'style' is deprecated and will be removed in version 1.3, use 'styles' instead.
  pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))


BokehModel(combine_events=True, render_bundle={'docs_json': {'a16f02bf-d959-4809-b7f3-f250cb924904': {'version…

C:\Users\wumin\AppData\Local\Temp\ipykernel_21240\1943865245.py:16: PanelDeprecationWarning: 'style' is deprecated and will be removed in version 1.3, use 'styles' instead.
  pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))


In [ ]:
import panel as pn  # GUI
pn.extension()

panels = [] # collect display 

context = [ {'role':'system', 'content':"""
You are OrderBot, an automated service to collect orders for a pizza restaurant. \
You first greet the customer, then collects the order, \
and then asks if it's a pickup or delivery. \
You wait to collect the entire order, then summarize it and check for a final \
time if the customer wants to add anything else. \
If it's a delivery, you ask for an address. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very conversational friendly style. \
The menu includes \
pepperoni pizza  12.95, 10.00, 7.00 \
cheese pizza   10.95, 9.25, 6.50 \
eggplant pizza   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
greek salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
sausage 3.00 \
canadian bacon 3.50 \
AI sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
bottled water 5.00 \
"""} ]  # accumulate messages

# 以下代码创建了一个基于Panel库的交互式对话界面，它允许用户输入文本消息并与一个名为button_conversation的按钮进行交互，以触发对话的进行和展示。
inp = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
# 创建了一个文本输入框控件（TextInput），称为inp。它的默认值设置为"Hi"，并且具有一个占位符文本，当用户没有输入时，显示 "Enter text here…"。这个控件用于用户输入消息。
button_conversation = pn.widgets.Button(name="Chat!")
# 创建了一个按钮控件（Button），称为button_conversation。按钮的标签（文本）设置为 "Chat!"。这个按钮用于触发对话的进行。
interactive_conversation = pn.bind(collect_messages, button_conversation)
# 使用pn.bind函数创建了一个可交互的对话。它将collect_messages函数与button_conversation按钮绑定在一起，以便在用户点击按钮时触发collect_messages函数并更新对话。

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)
# 创建了一个名为dashboard的列，将各个控件和对话部分组合到一起以构建完整的用户界面。
# inp 控件被添加到列中，用于用户输入消息。
# pn.Row(button_conversation) 创建一个包含按钮的水平行，并将它添加到列中，以便用户可以点击按钮来触发对话。
# pn.panel(interactive_conversation, loading_indicator=True, height=300) 创建一个包含交互式对话的面板，并将其添加到列中。这个面板使用了interactive_conversation来显示对话，并启用了加载指示器（loading indicator），这是在加载对话时显示的可视标识，以及设置了面板的高度为300像素。
dashboard

C:\Users\wumin\AppData\Local\Temp\ipykernel_21240\1943865245.py:16: PanelDeprecationWarning: 'style' is deprecated and will be removed in version 1.3, use 'styles' instead.
  pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))


BokehModel(combine_events=True, render_bundle={'docs_json': {'a16f02bf-d959-4809-b7f3-f250cb924904': {'version…

C:\Users\wumin\AppData\Local\Temp\ipykernel_21240\1943865245.py:16: PanelDeprecationWarning: 'style' is deprecated and will be removed in version 1.3, use 'styles' instead.
  pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))


In [12]:
messages =  context.copy()
# 创建了一个名为messages的列表，它复制了之前定义的context列表的内容。context列表包含了之前的用户和助手之间的对话消息。

messages.append(
{'role':'system', 'content':'create a json summary of the previous food order. Itemize the price for each item\
 The fields should be 1) pizza, include size 2) list of toppings 3) list of drinks, include size   4) list of sides include size  5)total price '},    
)
 #The fields should be 1) pizza, price 2) list of toppings 3) list of drinks, include size include price  4) list of sides include size include price, 5)total price '},    

response = get_completion_from_messages(messages, temperature=0)
print(response)


# 这段代码生成关于之前的食品订单的JSON摘要，包括不同项目的价格和详细信息。

Sure! Here's a JSON summary of your food order:

{
  "pizza": {
    "size": "12.95",
    "toppings": ["extra cheese", "mushrooms"]
  },
  "drinks": [
    {
      "name": "coke",
      "size": "3.00"
    },
    {
      "name": "sprite",
      "size": "2.00"
    }
  ],
  "sides": [
    {
      "name": "fries",
      "size": "4.50"
    }
  ],
  "total_price": "25.45"
}

Please let me know if there's anything else you'd like to add to your order!


## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!